# G2 D6-TV2 — Evaluation E1/E2
**GPU Colab: NOT RUN khi tạo notebook.** Chọn T4 hoặc GPU Colab Pro.
Nghiệm thu code (CLI/notebook/tests) độc lập với nghiệm thu thực nghiệm.
D6 không yêu cầu Adapter selection lock hoặc 72 run screening.
E2 chưa có checkpoint do chờ D10 joint selection/main training sẽ ghi **BLOCKED**;
E1 có thể đánh giá độc lập. So sánh đủ 8 category chỉ PASS khi có đủ 8 cặp hợp lệ.
Notebook không train; inference/metric/coverage dùng `scripts/eval_g2.py`.

Setup seed **42** dành cho phiên notebook. TRAIN/DEV seed, loss, threshold và budget
đọc từ checkpoint đã huấn luyện. Synthetic DEV dùng VALIDATION/good với loader,
protocol và seed đã khóa; không đọc TEST hoặc tạo dữ liệu thay thế.

Code cần có trên `g2/member2` trước khi chạy Colab. Nhập SHA ở `MSILA_COMMIT`, hoặc để trống
để chốt HEAD của branch một lần rồi checkout detached SHA. Dùng SHA đã lưu để lặp lại.
DINO checkout phải đúng revision đã dùng khi train.


## 1. Setup — GPU, repo, dependencies, Drive
Chạy từ đầu bằng **Run all** sau khi sửa các đường dẫn ở phần 2. Không bắt buộc riêng T4.
Venv giữ PyTorch CUDA của Colab nếu đã đáp ứng requirements.

In [ ]:
# @title Cell 1 — Setup
from pathlib import Path
import json, os, shutil, subprocess, sys
from IPython.display import HTML, Image as DisplayImage, display
from google.colab import drive

MSILA_COMMIT = '' # @param {type:'string'}
DINO_COMMIT = '6876159a11b4df116f30f667f8c9888617df0751' # @param {type:'string'}
SETUP_SEED = 42
WORK = Path('/content/msila_g2_evaluation')
REPO, DINO = WORK/'msila', WORK/'dinov3'
WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(['nvidia-smi'], check=True)
drive.mount('/content/drive')

def checkout(url, destination, branch, revision):
    if not (destination/'.git').is_dir():
        subprocess.run(['git', 'clone', '--depth', '1', '--branch', branch, url, str(destination)], check=True)
    if subprocess.check_output(['git', '-C', str(destination), 'status', '--porcelain'], text=True).strip():
        raise RuntimeError(f'BLOCKED: checkout có thay đổi: {destination}')
    if subprocess.check_output(['git', '-C', str(destination), 'remote', 'get-url', 'origin'], text=True).strip() != url:
        raise RuntimeError(f'BLOCKED: repo origin không đúng: {destination}')
    if not revision:
        revision = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    if len(revision) != 40 or any(c not in '0123456789abcdef' for c in revision):
        raise ValueError('BLOCKED: commit phải là SHA 40 ký tự')
    subprocess.run(['git', '-C', str(destination), 'fetch', '--depth', '1', 'origin', revision], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', revision], check=True)
    actual = subprocess.check_output(['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True).strip()
    if actual != revision:
        raise RuntimeError('BLOCKED: checkout SHA mismatch')
    return actual

MSILA_COMMIT = checkout('https://github.com/haduckien-raccoon/msila.git', REPO, 'g2/member2', MSILA_COMMIT)
DINO_COMMIT = checkout('https://github.com/facebookresearch/dinov3.git', DINO, 'main', DINO_COMMIT)
if not (REPO/'scripts/eval_g2.py').is_file():
    raise FileNotFoundError('BLOCKED: commit chưa chứa eval_g2.py; commit/push code D6-TV2 rồi chạy lại')
VENV = WORK/'venv'
if not (VENV/'bin/python').is_file():
    subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', str(VENV)], check=True)
PYTHON = str(VENV/'bin/python')
subprocess.run([PYTHON, '-m', 'pip', 'install', '-r', str(REPO/'requirements.txt')], check=True)
ENV = dict(os.environ, PYTHONHASHSEED=str(SETUP_SEED), PYTHONUNBUFFERED='1', MPLBACKEND='Agg',
           MPLCONFIGDIR=str(WORK/'matplotlib'), CUBLAS_WORKSPACE_CONFIG=':4096:8')
ENVIRONMENT = json.loads(subprocess.check_output([PYTHON, '-c',
    "import json,platform,torch; print(json.dumps(dict(python=platform.python_version(),torch=str(torch.__version__),cuda=torch.version.cuda,cuda_available=torch.cuda.is_available(),gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)))"],
    cwd=REPO, env=ENV, text=True))
if not ENVIRONMENT['cuda_available']:
    raise RuntimeError('BLOCKED: CUDA không khả dụng; chọn GPU runtime')
print(json.dumps(dict(environment=ENVIRONMENT, msila_commit=MSILA_COMMIT, dino_commit=DINO_COMMIT,
                     setup_seed=SETUP_SEED, gpu_evaluation='NOT RUN'), indent=2))
subprocess.run([PYTHON, 'scripts/eval_g2.py', '--help'], cwd=REPO, env=ENV, check=True)
sys.path.insert(0, str(REPO))
from scripts.eval_g2 import CATEGORIES, prepare_archives, copy_atomic, sync_tree, run_logged

## 2. Data / Checkpoint — preflight
Sửa đường dẫn Drive, tên 8 archive, `E1_ON_DRIVE`, `E2_ON_DRIVE`, pretrained weights.
Chọn `EXPERIMENTS = ['E1']` để đánh giá E1 riêng, hoặc `['E1', 'E2']` để so sánh.
**Copy đủ 8 `.tar.gz` từ Drive sang Colab trước, sau đó mới giải nén local.**
Chỉ giải nén TRAIN/good và VALIDATION/good. Synthetic DEV dùng protocol trong checkpoint.

Mỗi root checkpoint chứa `<category>/best.pt` và `best.pt.sha256` thật.
E1 có thể dùng checkpoint G1 cũ khi config/split/source hash khớp; nếu có `last.pt` và
`metrics.json`, evaluator kiểm tra thêm budget cuối. Thiếu checkpoint một category sẽ
ghi BLOCKED, vẫn tiếp tục các category/experiment khác. Không cần copy screening study
hay Adapter lock. Thiếu dataset/pretrained weights thì dừng với lỗi rõ ràng.
So sánh E1/E2 yêu cầu cùng backbone/decoder/seed/loss/budget và exact synthetic DEV.


In [ ]:
# @title Cell 2 — Data / Checkpoint
DRIVE_PROJECT = '/content/drive/MyDrive/[Q3-4] 2026/[S7] Computer Vision/CV-Nhóm 9' # @param {type:'string'}
BACKBONE = 'dinov3_vitb16' # @param {type:'string'}
PRETRAINED_ON_DRIVE = '' # @param {type:'string'}
E1_ON_DRIVE = '' # @param {type:'string'}
E2_ON_DRIVE = '' # @param {type:'string'}
EXPERIMENTS = ['E1', 'E2'] # Đổi thành ['E1'] khi E2 chưa train.
RUN_TAG = 'D6_E1_E2' # @param {type:'string'}
PROJECT = Path(DRIVE_PROJECT)
ARCHIVE_DIR = PROJECT/'data'
ARCHIVE_NAMES = {category: f'{category}.tar.gz' for category in CATEGORIES}
E1_SOURCE = Path(E1_ON_DRIVE) if E1_ON_DRIVE.strip() else PROJECT/'outputs/G1/E1'
E2_SOURCE = Path(E2_ON_DRIVE) if E2_ON_DRIVE.strip() else PROJECT/'outputs/G2/E2'
PRETRAINED_NAMES = {'dinov3_vits16':'dinov3_vits16_pretrain_lvd1689m-08c60483.pth',
                   'dinov3_vits16plus':'dinov3_vits16plus_pretrain_lvd1689m-4057cbaa.pth',
                   'dinov3_vitb16':'dinov3_vitb16_pretrain_lvd1689m-73cec8be.pth',
                   'dinov3_vitl16':'dinov3_vitl16_pretrain_lvd1689m-8aa4cbdd.pth',
                   'dinov3_vith16plus':'dinov3_vith16plus_pretrain_lvd1689m-7c1da9a5.pth'}
WEIGHT_SOURCE = Path(PRETRAINED_ON_DRIVE) if PRETRAINED_ON_DRIVE.strip() else PROJECT/'weights'/PRETRAINED_NAMES[BACKBONE]
OUTPUT = WORK/'outputs'/RUN_TAG/MSILA_COMMIT[:12]
DRIVE_OUTPUT = PROJECT/'outputs/G2/evaluation'/RUN_TAG/MSILA_COMMIT[:12]
OUTPUT.mkdir(parents=True, exist_ok=True)
sync_tree(DRIVE_OUTPUT, OUTPUT) # Resume sau khi Colab ngắt phiên.
SETUP_MANIFEST = dict(msila_commit=MSILA_COMMIT, dino_commit=DINO_COMMIT, setup_seed=SETUP_SEED,
                      environment=ENVIRONMENT, experiments=EXPERIMENTS,
                      original_e1_root=str(E1_SOURCE), original_e2_root=str(E2_SOURCE))
(OUTPUT/'notebook_environment.json').write_text(json.dumps(SETUP_MANIFEST, indent=2))
if not WEIGHT_SOURCE.is_file():
    (OUTPUT/'notebook_status.json').write_text(json.dumps(dict(status='BLOCKED', missing=[str(WEIGHT_SOURCE)])))
    sync_tree(OUTPUT, DRIVE_OUTPUT)
    raise FileNotFoundError(f'BLOCKED: thiếu pretrained checkpoint thật: {WEIGHT_SOURCE}')
try:
    DATA = prepare_archives(ARCHIVE_DIR, ARCHIVE_NAMES, WORK/'archives', WORK/'dataset')
except (OSError, ValueError, RuntimeError) as exc:
    (OUTPUT/'notebook_status.json').write_text(json.dumps(dict(status='BLOCKED', reason=str(exc))))
    sync_tree(OUTPUT, DRIVE_OUTPUT)
    raise
WEIGHTS = copy_atomic(WEIGHT_SOURCE, WORK/'weights'/WEIGHT_SOURCE.name)
E1_ROOT, E2_ROOT = WORK/'checkpoints/E1', WORK/'checkpoints/E2'
for experiment, source, target in [('E1',E1_SOURCE,E1_ROOT),('E2',E2_SOURCE,E2_ROOT)]:
    if target.exists():
        shutil.rmtree(target) # Không dùng checkpoint cũ khi đường dẫn Drive thay đổi.
    if experiment in EXPERIMENTS and source.is_dir():
        sync_tree(source, target)
    if experiment in EXPERIMENTS:
        missing = [category for category in CATEGORIES if not (target/category/'best.pt').is_file()]
        print(experiment, 'checkpoint thiếu (CLI sẽ ghi BLOCKED):', missing)
COMMON_ARGS = ['--experiments',*EXPERIMENTS,'--e1-root',str(E1_ROOT),'--e2-root',str(E2_ROOT),
               '--data-root',str(DATA),'--repo-dir',str(DINO),'--weights',str(WEIGHTS),
               '--device','cuda','--output-root',str(OUTPUT)]

def run_eval(arguments, stage):
    return run_logged([PYTHON,'scripts/eval_g2.py',*arguments], cwd=REPO, output_root=OUTPUT,
                      drive_output=DRIVE_OUTPUT, stage=stage, env=ENV)

PREFLIGHT_CODE = run_eval([*COMMON_ARGS,'--categories','all','--preflight'], 'preflight')
if PREFLIGHT_CODE == 1:
    raise RuntimeError('FAIL: xem preflight/coverage.json và preflight.log trên Drive')
print('Preflight exit:', PREFLIGHT_CODE, '(0=READY; 2=BLOCKED cho input thiếu). Tiếp tục E1 nếu có checkpoint.')


## 3. Smoke — một category, ba mẫu DEV gốc
Kiểm tra restore heads, extraction, native map, tiny/mixed AU-PRO, Dice và P99.
Output riêng `smoke/`; **SMOKE_PASS không được tính vào coverage nghiệm thu**.

In [ ]:
# @title Cell 3 — Smoke test
SMOKE_CATEGORY = 'rice' # @param {type:'string'}
SMOKE_CODE = run_eval([*COMMON_ARGS,'--categories',SMOKE_CATEGORY,'--smoke'], 'smoke')
if SMOKE_CODE == 1:
    raise RuntimeError('FAIL: xem smoke/summary.json và smoke.log trên Drive')
print('Smoke exit:', SMOKE_CODE, '(0=SMOKE_PASS; 2=BLOCKED). Smoke không đóng góp vào coverage nghiệm thu.')


## 4. Full evaluation — experiment đã chọn, tám category
`--resume` chỉ skip kết quả còn hợp lệ: checkpoint/config/source/code hash, metric,
QA và toàn bộ native map/mask đã lưu. Run thiếu/khác inputs sẽ không được tính PASS.
Muốn chạy riêng một category: đổi `FULL_CATEGORIES`; vẫn giữ đủ 8 hàng báo cáo,
không có macro 8 category khi coverage chưa đủ. Đồng bộ Drive mỗi 30 giây và cuối run.

In [ ]:
# @title Cell 4 — Full evaluation / resume
FULL_CATEGORIES = 'all' # @param {type:'string'}
FULL_CODE = run_eval([*COMMON_ARGS,'--categories',FULL_CATEGORIES,'--resume'], 'full')
print('Evaluator exit:', FULL_CODE, '(0=run đã chọn PASS; 1=FAIL; 2=BLOCKED). Xem kết quả thực ở phần 5.')

## 5. Results — bảng, biểu đồ, native maps và export
Không tính lại metric trong notebook. Bảng CSV luôn giữ đủ 8 category với ô trống cho run thiếu.
Macro chỉ xuất khi đủ 8 cặp có cùng protocol và cùng exact synthetic DEV.
Toàn bộ CSV/JSON/native `.npy`/examples/config/Git/GPU/log được đồng bộ Google Drive.

In [ ]:
# @title Cell 5 — Results / Export
import csv, html
REPORT_CODE = run_eval([*COMMON_ARGS,'--categories','all','--report-only'], 'report')
summary = json.loads((OUTPUT/'summary.json').read_text())
print(json.dumps({k:v for k,v in summary.items() if k != 'records'}, indent=2))
with (OUTPUT/'metrics_E1_E2.csv').open(newline='') as handle:
    rows = list(csv.DictReader(handle))
columns = ['category','E1_status','E2_status','E1_synthetic_dev_aupro_0_05',
           'E2_synthetic_dev_aupro_0_05','E2_minus_E1_aupro_0_05','pair_status']
table = '<table><tr>'+''.join('<th>'+html.escape(c)+'</th>' for c in columns)+'</tr>'
table += ''.join('<tr>'+''.join('<td>'+html.escape(row.get(c,''))+'</td>' for c in columns)+'</tr>' for row in rows)+'</table>'
display(HTML(table))
display(DisplayImage(filename=str(OUTPUT/'E2_minus_E1_aupro.png')))
for experiment in ('E1','E2'):
    examples = sorted((OUTPUT/experiment).glob('*/examples/*/comparison.png'))
    for example in examples[:2]:
        print(experiment, example.parent.name, 'Original / Synthetic input / Exact GT / Probability / Overlay')
        display(DisplayImage(filename=str(example)))
if summary['status'] == 'PASS' and summary['complete'] and summary['paired_pass'] == 8:
    print('PASS: E1 8/8, E2 8/8, paired 8/8. Macro:', summary['macro_mean_8categories'])
else:
    print('Run đã chọn:', summary['status'], '| So sánh E1/E2:', summary['comparison_status'])
    print('Coverage GPU thật:', summary['coverage'], '| paired:', summary['paired_pass'], '/8')
    print('Macro từng experiment (chỉ khi đủ 8):', summary['per_experiment_macro_mean_8categories'])
    print('So sánh đủ 8 category chưa PASS; xem coverage.json để biết run BLOCKED/NOT RUN.')
sync_tree(OUTPUT, DRIVE_OUTPUT)
print('Đã lưu:', DRIVE_OUTPUT)